# 🎬 News Video Pipeline — Colab GPU Worker (GGUF / VRAM自動対応)

Mac 側 (`cli.ts --remote`) と **Google Drive フォルダ**で連携する GPU ワーカー。

- **入力** (Drive `news-video-pipeline/inbox/<jobId>/`): `script.txt`, `face.*`, `job.json`, (任意)`voice_ref.wav`
- **処理**: [1] Fishaudio で音声 → [2] LTX-2.3 (Lipdub LoRA) でリップシンク動画
- **出力** (Drive `news-video-pipeline/outbox/<jobId>/`): `narration.wav`, `talking.mp4`, `done.json`

**GPU別の量子化（自動選択）**: T4(16GB)→Q3_K_M / L4(24GB)→Q5_K_M / A100(40GB+)→Q8_0。
※ LTX-2.3 22B のフル(bf16)は 42GB で A100-40GB でも溢れるため、GGUF が前提。
Drive マイドライブ直下に `news-video-pipeline` フォルダがある状態（Mac の rclone remote と同じ場所）。


In [ ]:
# === 1. Drive マウント・バス設定・VRAM判定 ================================
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, torch
BUS   = pathlib.Path('/content/drive/MyDrive/news-video-pipeline')
INBOX = BUS / 'inbox'; OUTBOX = BUS / 'outbox'; MODELS = BUS / 'models_cache'
for p in (INBOX, OUTBOX, MODELS): p.mkdir(parents=True, exist_ok=True)

vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if   vram < 18: QUANT = 'Q3_K_M'   # T4 16GB
elif vram < 26: QUANT = 'Q5_K_M'   # L4 24GB
else:           QUANT = 'Q8_0'     # A100 40/80GB
print(f'GPU VRAM ≈ {vram:.0f}GB → 量子化 {QUANT}')
print('BUS =', BUS)

In [ ]:
# === 2. ComfyUI + LTXVideo + GGUF ノード + cloudflared =====================
%cd /content
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt
%cd /content/ComfyUI/custom_nodes
![ -d ComfyUI-LTXVideo ] || git clone https://github.com/Lightricks/ComfyUI-LTXVideo
![ -d ComfyUI-GGUF ]     || git clone https://github.com/city96/ComfyUI-GGUF
!pip -q install -r ComfyUI-LTXVideo/requirements.txt
!pip -q install -r ComfyUI-GGUF/requirements.txt
!pip -q install -U huggingface_hub hf_transfer
# 動画書き出し用にこのカスタムノードもよく使う
![ -d ComfyUI-VideoHelperSuite ] || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite
!pip -q install -r ComfyUI-VideoHelperSuite/requirements.txt 2>/dev/null
# cloudflared（手動GUIで初回ワークフロー確認用）
![ -f /usr/local/bin/cloudflared ] || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)
print('nodes ready')

In [ ]:
# === 3. GGUF モデル取得（Drive キャッシュ → ComfyUI へ symlink）===========
try:
    import hf_transfer  # noqa: F401
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
except Exception:
    os.environ.pop('HF_HUB_ENABLE_HF_TRANSFER', None)
from huggingface_hub import hf_hub_download

CK = pathlib.Path('/content/ComfyUI/models')
DIFF = f'ltx-2.3-22b-dev-{QUANT}.gguf'

# (repo_id, filename, ComfyUI サブフォルダ)
FILES = [
    ('unsloth/LTX-2.3-GGUF',            DIFF,                                                   'unet'),
    ('unsloth/gemma-3-12b-it-qat-GGUF', 'gemma-3-12b-it-qat-UD-Q4_K_XL.gguf',                   'text_encoders'),
    ('unsloth/gemma-3-12b-it-qat-GGUF', 'mmproj-BF16.gguf',                                     'text_encoders'),
    ('unsloth/LTX-2.3-GGUF',            'vae/ltx-2.3-22b-dev_video_vae.safetensors',            'vae'),
    ('unsloth/LTX-2.3-GGUF',            'vae/ltx-2.3-22b-dev_audio_vae.safetensors',            'vae'),
    ('unsloth/LTX-2.3-GGUF',            'text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors', 'text_encoders'),
    ('Lightricks/LTX-2.3',             'ltx-2.3-22b-ic-lora-lipdub-0.9.safetensors',           'loras'),
    ('Lightricks/LTX-2.3',             'ltx-2.3-22b-distilled-lora-384.safetensors',           'loras'),
]
for repo, fn, sub in FILES:
    dst_dir = CK / sub; dst_dir.mkdir(parents=True, exist_ok=True)
    base = pathlib.Path(fn).name
    cached = MODELS / base
    if not cached.exists():
        print('↓', repo, fn)
        p = hf_hub_download(repo_id=repo, filename=fn, local_dir=str(MODELS))
        # サブフォルダ付きDLはネスト保存されるので基底名に寄せる
        if pathlib.Path(p) != cached:
            os.replace(p, cached)
    link = dst_dir / base
    if not link.exists():
        os.symlink(cached, link)
    print('  →', link)
print('✅ models linked. unet =', DIFF)

In [ ]:
# === 4. Fishaudio (fish-speech) =========================================
%cd /content
![ -d fish-speech ] || git clone https://github.com/fishaudio/fish-speech
%cd /content/fish-speech
!pip -q install -e .
from huggingface_hub import snapshot_download
FISH_CKPT = MODELS / 'fish-speech-1.5'
if not FISH_CKPT.exists():
    snapshot_download(repo_id='fishaudio/fish-speech-1.5', local_dir=str(FISH_CKPT))
print('fish-speech ckpt:', FISH_CKPT)

In [ ]:
# === 5. TTS 関数（Fishaudio）============================================
# ⚠️ fish-speech はバージョンで CLI 引数が変わる。エラーが出たらここを調整。
import subprocess, pathlib

def tts_fish(text: str, out_wav: str, voice_ref: str | None = None):
    work = pathlib.Path('/content/_tts'); work.mkdir(exist_ok=True)
    ckpt = str(FISH_CKPT)
    gen = ['python', '-m', 'tools.llama.generate', '--text', text,
           '--checkpoint-path', ckpt, '--num-samples', '1', '--output-dir', str(work)]
    if voice_ref:
        gen += ['--prompt-text', text, '--prompt-tokens', voice_ref]
    subprocess.run(gen, cwd='/content/fish-speech', check=True)
    subprocess.run(['python', '-m', 'tools.vqgan.inference', '-i', str(work / 'codes_0.npy'),
                    '--checkpoint-path', ckpt, '-o', out_wav],
                   cwd='/content/fish-speech', check=True)
    return out_wav
# 動作確認: tts_fish('こんばんは。AIニュースの時間です。', '/content/test.wav')

In [ ]:
# === 6. ComfyUI 起動 + cloudflared（初回はGUIでワークフローを確認）=========
import subprocess, time, requests, re, pathlib
PORT = 8188
comfy = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', str(PORT)],
                         cwd='/content/ComfyUI')
for _ in range(90):
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/system_stats').ok: break
    except Exception: pass
    time.sleep(2)
print('ComfyUI up')

# 公開URL（GUIで lipdub ワークフローを開いて確認・API書き出しするため）
logf = open('/content/cf.log', 'w')
cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
                      stdout=logf, stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m: url = m.group(0); break
print('🌐 ComfyUI GUI:', url)
print(("""
=================================================
初回だけ GUI で以下を実施（自動化の土台作り）:
1) 上の URL を開く
2) Manager か Workflow メニューから LTX-2.3 の
   「Image+Audio to Video / Lipdub」テンプレートを開く
3) モデルローダを GGUF 用に: 「Unet Loader (GGUF)」で {DIFF} を選択、
   text encoder=gemma GGUF, vae=video/audio vae, lora=lipdub を接続
4) 画像と音声を 1 度入れて生成し、口が動くのを確認
5) メニュー → Save (API Format) で JSON を保存し、
   Drive の news-video-pipeline/lipdub_api.json に置く
   （この JSON を下のワーカーが使い回す）
=================================================""").replace('{DIFF}', DIFF))

In [ ]:
# === 7. リップシンク関数（保存した API ワークフローを使う）=================
import json, shutil, uuid, time, requests, pathlib

COMFY_INPUT = pathlib.Path('/content/ComfyUI/input'); COMFY_INPUT.mkdir(exist_ok=True)
API_WF = BUS / 'lipdub_api.json'   # 手順6で保存した API 形式ワークフロー

def _queue_and_wait(prompt: dict, timeout=1800):
    cid = uuid.uuid4().hex
    r = requests.post(f'http://127.0.0.1:{PORT}/prompt', json={'prompt': prompt, 'client_id': cid})
    r.raise_for_status(); pid = r.json()['prompt_id']
    t0 = time.time()
    while time.time() - t0 < timeout:
        h = requests.get(f'http://127.0.0.1:{PORT}/history/{pid}').json()
        if pid in h: return h[pid]
        time.sleep(3)
    raise TimeoutError('ComfyUI render timed out')

def _patch_inputs(prompt: dict, image_name: str, audio_name: str):
    for nid, node in prompt.items():
        ct = node.get('class_type', ''); ins = node.setdefault('inputs', {})
        if ct == 'LoadImage':                      ins['image'] = image_name
        if ct in ('LoadAudio', 'VHS_LoadAudio'):   ins['audio'] = audio_name
    return prompt

def lipsync_ltx(image_path: str, audio_path: str, out_mp4: str):
    if not API_WF.exists():
        raise FileNotFoundError(f'{API_WF} が無い。手順6でGUIからAPI形式を保存してください')
    img = 'face' + pathlib.Path(image_path).suffix
    shutil.copy(image_path, COMFY_INPUT / img)
    shutil.copy(audio_path, COMFY_INPUT / 'narration.wav')
    prompt = _patch_inputs(json.loads(API_WF.read_text()), img, 'narration.wav')
    hist = _queue_and_wait(prompt)
    for nid, out in hist.get('outputs', {}).items():
        for key in ('gifs', 'videos', 'images'):
            for f in out.get(key, []):
                src = pathlib.Path('/content/ComfyUI/output') / f.get('subfolder', '') / f['filename']
                if src.suffix.lower() in ('.mp4', '.webm', '.mov'):
                    shutil.copy(src, out_mp4); return out_mp4
    raise RuntimeError('出力動画が見つからない（保存ノードを確認）')

In [ ]:
# === 8. ワーカーループ（Drive inbox を監視）==============================
import time, json, pathlib

def process_job(job_dir: pathlib.Path):
    spec = json.loads((job_dir / 'job.json').read_text())
    jid = spec['jobId']; print('▶ job', jid)
    out = OUTBOX / jid; out.mkdir(parents=True, exist_ok=True)
    text = (job_dir / spec['script']).read_text(encoding='utf-8')
    face = job_dir / spec['face']
    ref  = job_dir / spec['voiceRef'] if spec.get('voiceRef') else None
    wav = str(out / 'narration.wav')
    tts_fish(text, wav, str(ref) if ref else None)              # [1] 音声
    lipsync_ltx(str(face), wav, str(out / 'talking.mp4'))        # [2] リップシンク
    (out / 'done.json').write_text(json.dumps({'jobId': jid, 'ok': True}))  # 最後に書く
    print('✓ done', jid)

def worker_loop(poll=15):
    print('👀 inbox 監視開始 ...', INBOX); seen = set()
    while True:
        for job_dir in sorted(INBOX.glob('*')):
            jid = job_dir.name
            if jid in seen or not (job_dir / 'job.json').exists(): continue
            if (OUTBOX / jid / 'done.json').exists(): seen.add(jid); continue
            try: process_job(job_dir); seen.add(jid)
            except Exception as e:
                print('✗ failed', jid, e)
                (OUTBOX / jid).mkdir(parents=True, exist_ok=True)
                (OUTBOX / jid / 'error.json').write_text(json.dumps({'error': str(e)})); seen.add(jid)
        time.sleep(poll)

worker_loop()